# Conv_Emergency ↔ source(PREDICTIVE/DIAGNOSTIC) 및 Pour_Zone_CNT PV 리셋 검증

지난 3일치 아래 3개 태그를 받아 세 가지를 확인한다.

- `Conv_Emergency`
- `Pour_Zone_CNT PV`
- `Conv_Brake_Time`

(`source` 컴럼은 태그로 요청하지 않아도 `Log_Extractor` 가 자동으로 함께 붙여준다.)

**1) 리셋 검증** — `Pour_Zone_CNT PV` 가 음수 diff(리셋)가 난 다음, 그 직후 값들이 정상 구간과 비슷하게 부드럽게 이어지는지(리셋 직전 값만 버려도 되는지), 아니면 뒤가 튜는지(더 깊이 버려야 하는지) 확인.

**2) Conv_Emergency ↔ source** — Conv_Emergency 가 켜질 때 source 가 DIAGNOSTIC 으로 바뀌는지, 그리고 source 가 바뀜 때 Conv_Emergency 와 무관한(다른 원인의) 경우가 몇 번이나 되는지.

**3) source 전환 시점의 변화** — source 가 바뀜 순간 `Pour_Zone_CNT PV` 와 `Conv_Brake_Time` 이 어떻게 변하는지 산점도/히스토그램으로 확인.

## 1. 모듈 로드 및 파라미터

In [ ]:
import os
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
%matplotlib inline
matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

# 이 노트북은 parkkt/conv/ 에 있고, Log_Extractor.py 와 .env 는 레포 루트에 있다.
ROOT_DIR = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.insert(0, ROOT_DIR)

from Log_Extractor import LogExtractor

ENV_PATH = os.path.join(ROOT_DIR, ".env")
SAVE_DIR = os.path.join(os.getcwd(), "extracted_csv")

pd.set_option("display.width", 200)
print(f"ROOT_DIR = {ROOT_DIR}")
print(f"ENV_PATH = {ENV_PATH}  (존재={os.path.exists(ENV_PATH)})")
print(f"SAVE_DIR = {SAVE_DIR}")

In [ ]:
TAGS = [
    "Conv_Emergency",
    "Pour_Zone_CNT PV",
    "Conv_Brake_Time",
]

START_TIME = "-3d"
END_TIME = "now()"
FILE_PREFIX = "Conv_Emergency_Source"

# Conv_Emergency 변화와 source 전환이 같은 사건으로 보기에 허용할 시간차.
# 실측: Conv_Emergency ON 후 약 0.6~0.8초 뒤에 source 가 바뀌는 패턴이 보여 3초로 넣는다.
MATCH_WINDOW_SEC = 3

## 2. 추출 및 저장

In [ ]:
extractor = LogExtractor(env_path=ENV_PATH)
raw = extractor.get_data(start_time=START_TIME, end_time=END_TIME, target_tags=TAGS)
print(f"{raw.shape[0]:,}행 × {raw.shape[1]}컴럼   기간: {raw.index.min()} ~ {raw.index.max()}")
raw["source"].value_counts()

In [ ]:
if raw.empty:
    print("⚠️ 저장할 데이터가 없습니다.")
else:
    os.makedirs(SAVE_DIR, exist_ok=True)
    timestamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")
    raw_path = os.path.join(SAVE_DIR, f"{FILE_PREFIX}_raw_{timestamp}.csv")
    raw.to_csv(raw_path, encoding="utf-8-sig")
    print(f"💾 원본: {raw_path}")

## 3. Part 1 — `Pour_Zone_CNT PV` 리셋(음수 diff) 검증

이전 분석에서는 리셋(음수 diff) 자체만 즜상율 계산에서 제외했다. 여기서는 한 단계 더 나가서,
**리셋 직후의 값들이 얼마나 정상적으로 이어지는지**를 확인한다. 만약 리셋 직후 스텝이 평소보다
훨씬 크거나 불규칙하다면, 단순히 음수 diff 한 줄만 지우는 것으로는 부족하다는 증거가 된다.

In [ ]:
raw_diff = raw["Pour_Zone_CNT PV"].diff()
reset = raw_diff < 0
print(f"리셋(음수 diff) {int(reset.sum()):,}회 / 전체 diff {int(raw_diff.notna().sum()):,}회")

reset_pos = np.flatnonzero(reset.to_numpy())
reset_pos = reset_pos[reset_pos + 1 < len(raw_diff)]
first_after = raw_diff.iloc[reset_pos + 1]

normal_step = raw_diff[raw_diff > 0]
print("\n정상 스텝(0 초과, 리셋 아님) 분포:")
print(normal_step.describe())
print("\n리셋 직후 바로 다음 값의 diff 분포 (0이하 포함):")
print(first_after.describe())

first_after_pos = first_after[first_after > 0]
ks = stats.ks_2samp(normal_step.dropna(), first_after_pos.dropna())
print(f"\nKS 검정(정상 스텝 vs 리셋직후 첫스텝, 둘 다 양수만): stat={ks.statistic:.4f}  p={ks.pvalue:.2g}")

odd_after = int((first_after <= 0).sum())
print(f"리셋 직후 바로 다음 값이 무변화/추가리셋인 경우: {odd_after:,}/{len(first_after):,} ({odd_after / len(first_after) * 100:.1f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
bins = np.arange(0, max(normal_step.max(), first_after_pos.max()) + 2)
ax.hist(normal_step, bins=bins, alpha=0.5, density=True, label="정상 스텝", color="steelblue")
ax.hist(first_after_pos, bins=bins, alpha=0.5, density=True, label="리셋 직후 첫 스텝", color="darkorange")
ax.set_xlabel("스텝 크기")
ax.set_ylabel("밀도")
ax.set_title("정상 스텝 vs 리셋 직후 첫 스텝 분포")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

In [ ]:
n_show = 6
fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True)
for ax, rp in zip(axes.flat, reset_pos[:n_show]):
    window = raw["Pour_Zone_CNT PV"].iloc[max(0, rp - 1): rp + 5].to_numpy()
    xs = range(-1, len(window) - 1)
    ax.plot(xs, window, marker="o")
    ax.axvline(0, color="red", linestyle="--", linewidth=1)
    ax.set_title(str(raw.index[rp])[:19], fontsize=8)
    ax.grid(alpha=0.3)
fig.suptitle("리셋 전후 실제 값 흐름 (샘플, x=0 이 리셋 직후값)")
fig.tight_layout()

## 4. Part 2 — `Conv_Emergency` ↔ `source` 전환 관계

In [ ]:
source_change = raw["source"] != raw["source"].shift(1)
source_change.iloc[0] = False
to_diag = source_change & (raw["source"] == "DIAGNOSTIC")
to_pred = source_change & (raw["source"] == "PREDICTIVE")

em = raw["Conv_Emergency"]
em_on = (em == 1) & (em.shift(1) != 1)
em_on.iloc[0] = False
em_off = (em != 1) & (em.shift(1) == 1)
em_off.iloc[0] = False

diag_times = raw.index[to_diag]
pred_times = raw.index[to_pred]
on_times = raw.index[em_on]
off_times = raw.index[em_off]

print(f"source → DIAGNOSTIC 전환: {len(diag_times)}건")
print(f"source → PREDICTIVE 복귀: {len(pred_times)}건")
print(f"Conv_Emergency ON: {len(on_times)}건 / OFF: {len(off_times)}건")

In [ ]:
def nearest_gap_sec(t, target_times):
    if len(target_times) == 0:
        return np.inf
    return np.abs((target_times - t).total_seconds()).min()


def match_summary(event_times, target_times, window_sec, event_label, target_label):
    matched = sum(nearest_gap_sec(t, target_times) <= window_sec for t in event_times)
    print(f"{event_label} {len(event_times)}건 중 {matched}건이 ±{window_sec}초 이내 {target_label} 이벤트와 매칭, "
          f"{len(event_times) - matched}건 불일치")
    return matched


print("--- Conv_Emergency 기준으로 source 전환이 따라오는지 ---")
match_summary(on_times, diag_times, MATCH_WINDOW_SEC, "Conv_Emergency ON", "DIAGNOSTIC 전환")
match_summary(off_times, pred_times, MATCH_WINDOW_SEC, "Conv_Emergency OFF", "PREDICTIVE 복귀")

print("\n--- 거꾸로, source 전환 중 Conv_Emergency 로 설명되는 건수 ---")
m_diag = match_summary(diag_times, on_times, MATCH_WINDOW_SEC, "DIAGNOSTIC 전환", "Conv_Emergency ON")
m_pred = match_summary(pred_times, off_times, MATCH_WINDOW_SEC, "PREDICTIVE 복귀", "Conv_Emergency OFF")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
labels = [f"DIAGNOSTIC 전환\n(총 {len(diag_times)}건)", f"PREDICTIVE 복귀\n(총 {len(pred_times)}건)"]
emergency_part = [m_diag, m_pred]
other_part = [len(diag_times) - m_diag, len(pred_times) - m_pred]

x = np.arange(len(labels))
ax.bar(x, emergency_part, label="Conv_Emergency 매칭", color="crimson")
ax.bar(x, other_part, bottom=emergency_part, label="기타 원인", color="lightgray")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("건수")
ax.set_title(f"source 전환 원인 분해 (±{MATCH_WINDOW_SEC}초 매칭 기준)")
ax.legend()
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()

In [ ]:
if len(on_times):
    t0 = on_times[0]
    lo, hi = t0 - pd.Timedelta(seconds=20), t0 + pd.Timedelta(seconds=20)
    w = raw.loc[lo:hi].copy()
    w["source_num"] = (w["source"] == "DIAGNOSTIC").astype(int)

    fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
    axes[0].step(w.index, w["Conv_Emergency"], where="post", color="red")
    axes[0].set_ylabel("Conv_Emergency")
    axes[1].step(w.index, w["source_num"], where="post", color="purple")
    axes[1].set_ylabel("source\n(1=DIAGNOSTIC)")
    axes[2].step(w.index, w["Conv_Brake_Time"], where="post", color="steelblue")
    axes[2].set_ylabel("Conv_Brake_Time")
    axes[3].step(w.index, w["Pour_Zone_CNT PV"], where="post", color="green")
    axes[3].set_ylabel("Pour_Zone_CNT PV")
    for ax in axes:
        ax.axvline(t0, color="black", linestyle="--", linewidth=1)
        ax.grid(alpha=0.3)
    axes[-1].set_xlabel("시각")
    fig.suptitle(f"Conv_Emergency ON 시점 전후 40초 ({t0})")
    fig.autofmt_xdate()
    fig.tight_layout()
else:
    print("Conv_Emergency ON 이벤트가 없습니다.")

## 5. Part 3 — `source` 전환 시점의 `Pour_Zone_CNT PV` / `Conv_Brake_Time` 변화

In [ ]:
sc_pos = np.flatnonzero(source_change.to_numpy())
sc_pos = sc_pos[sc_pos > 0]

trans = pd.DataFrame({
    "시각": raw.index[sc_pos],
    "방향": np.where(raw["source"].iloc[sc_pos].to_numpy() == "DIAGNOSTIC", "→DIAGNOSTIC", "→PREDICTIVE"),
    "brake_전": raw["Conv_Brake_Time"].iloc[sc_pos - 1].to_numpy(),
    "brake_후": raw["Conv_Brake_Time"].iloc[sc_pos].to_numpy(),
    "cnt_전": raw["Pour_Zone_CNT PV"].iloc[sc_pos - 1].to_numpy(),
    "cnt_후": raw["Pour_Zone_CNT PV"].iloc[sc_pos].to_numpy(),
})
trans["brake_변화"] = trans["brake_후"] - trans["brake_전"]
trans["cnt_변화"] = trans["cnt_후"] - trans["cnt_전"]
trans.groupby("방향")[["brake_변화", "cnt_변화"]].describe().T

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
colors = {"→DIAGNOSTIC": "crimson", "→PREDICTIVE": "steelblue"}

for direction, color in colors.items():
    d = trans[trans["방향"] == direction]
    axes[0, 0].scatter(d["brake_전"], d["brake_후"], alpha=0.5, label=direction, color=color, s=15)
    axes[0, 1].scatter(d["cnt_전"], d["cnt_후"], alpha=0.5, label=direction, color=color, s=15)

lims_b = [trans[["brake_전", "brake_후"]].min().min(), trans[["brake_전", "brake_후"]].max().max()]
axes[0, 0].plot(lims_b, lims_b, color="gray", linestyle="--", linewidth=1, label="변화없음 기준선")
axes[0, 0].set_xlabel("전환 전 Conv_Brake_Time")
axes[0, 0].set_ylabel("전환 후 Conv_Brake_Time")
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(alpha=0.3)

lims_c = [trans[["cnt_전", "cnt_후"]].min().min(), trans[["cnt_전", "cnt_후"]].max().max()]
axes[0, 1].plot(lims_c, lims_c, color="gray", linestyle="--", linewidth=1, label="변화없음 기준선")
axes[0, 1].set_xlabel("전환 전 Pour_Zone_CNT PV")
axes[0, 1].set_ylabel("전환 후 Pour_Zone_CNT PV")
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(alpha=0.3)

axes[1, 0].hist([trans[trans["방향"] == d]["brake_변화"] for d in colors],
                bins=20, label=list(colors.keys()), color=list(colors.values()), alpha=0.7)
axes[1, 0].set_xlabel("Conv_Brake_Time 변화(후-전)")
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(alpha=0.3)

axes[1, 1].hist([trans[trans["방향"] == d]["cnt_변화"] for d in colors],
                bins=20, label=list(colors.keys()), color=list(colors.values()), alpha=0.7)
axes[1, 1].set_xlabel("Pour_Zone_CNT PV 변화(후-전)")
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid(alpha=0.3)

fig.suptitle("source 전환 시점의 Conv_Brake_Time / Pour_Zone_CNT PV 변화")
fig.tight_layout()

## 6. 저장

In [ ]:
if trans.empty:
    print("⚠️ source 전환 이벤트가 없어 저장할 데이터가 없습니다.")
else:
    timestamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")
    trans_path = os.path.join(SAVE_DIR, f"{FILE_PREFIX}_전환이벤트_{timestamp}.csv")
    trans.to_csv(trans_path, index=False, encoding="utf-8-sig")
    print(f"💾 source 전환 이벤트: {trans_path}")